In [1]:
from core.retro_q import RetroQ
from core.expectation import Expectation
import pennylane as qp
import numpy as np
from qiskit import QuantumCircuit


PennyLane Bug: Crash in Program with Cancel Inverses

In [2]:
def target_function():
    U = 1 / np.sqrt(2) * np.array([[1, 1], [1, -1]])
    qp.QubitUnitary(U, 0)
    qp.adjoint(qp.QubitUnitary(U, 0))

exp = Expectation(gate_count='decrease')

rq = RetroQ(target_function, compiler_pass=qp.transforms.cancel_inverses, expectation=exp, framework='pennylane')

results = rq.run()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

PennyLane Bug: Global Phase and Controlled Operator

In [3]:
def target_function():
    qp.Hadamard(wires=0)
    qp.RZ(0.1, wires=0)

exp = Expectation( gate_count='unchanged')

rq = RetroQ(target_function, compiler_pass=qp.transforms.single_qubit_fusion,  compiler_pass_kwargs={'exclude_gates':['Hadamard']}, expectation=exp, framework='pennylane')

results = rq.run()
results

{'stage_1_differences': [{'detail': 'compiler_pass(adjoint(func))',
   'differences': {'gate_count': {'baseline': 4, 'actual': 3, 'ok': False},
    'expectation_value': {'baseline': [np.float64(0.9999999999999996)],
     'actual': [np.float64(1.0)],
     'ok': array([ True])}}}],
 'stage_2_differences': [{'detail': 'control(compiler_pass(adjoint(func))',
   'differences': {'gate_count': {'baseline': 6, 'actual': 6, 'ok': True},
    'expectation_value': {'baseline': np.float64(0.9999999999999996),
     'actual': np.float64(-0.9999999999999996),
     'ok': np.False_}}}]}

PennyLane Bugs - Adjoint of Quantum Operation not Considered in Pattern Matching Optimization

In [4]:
def target_function():
    qp.Hadamard(0)
    qp.PauliZ(0)
    qp.Hadamard(0)
    qp.RZ(0.1, wires=0)

exp = Expectation(gate_count='decrease')

ops = [qp.Hadamard(0), qp.PauliZ(0), qp.Hadamard(0), qp.PauliX(0)]
pattern = qp.tape.QuantumTape(ops)

rq = RetroQ(target_function, compiler_pass=qp.transforms.pattern_matching_optimization, compiler_pass_kwargs={'pattern_tapes':[pattern]}, expectation=exp, framework='pennylane')

results = rq.run()
results

{'stage_1_differences': [{'detail': 'compiler_pass(adjoint(func))',
   'differences': {'gate_count': {'baseline': 8, 'actual': 8, 'ok': False},
    'expectation_value': {'baseline': [np.float64(0.9999999999999993)],
     'actual': [np.float64(0.9999999999999993)],
     'ok': array([ True])}}}],
 'stage_2_differences': [{'detail': 'control(compiler_pass(adjoint(func))',
   'differences': {'gate_count': {'baseline': 10, 'actual': 10, 'ok': False},
    'expectation_value': {'baseline': np.float64(0.9999999999999996),
     'actual': np.float64(0.9999999999999993),
     'ok': np.True_}}}]}

Qiskit Bugs: Incorrect Commutation Inverse Cancellation with SWAP
and Controlled-RZ Gates

In [5]:
from qiskit.transpiler.passes import CommutativeInverseCancellation

def target_function():
    qc = QuantumCircuit(2)
    qc.swap(0, 1)
    qc.crz(3.2, 0, 1)
    qc.swap(0, 1)
    return qc    

# 2. Define user expectations for the compiler pass
exp = Expectation( gate_count='unchanged')

# 3. Initialize RetroQ with the pass to test
rq =  RetroQ(target_function, compiler_pass=CommutativeInverseCancellation, expectation=exp, framework='qiskit')
results = rq.run()
results 

{'stage_1_differences': [{'detail': 'compiler_pass(adjoint(func))',
   'differences': {'gate_count': {'baseline': 6, 'actual': 4, 'ok': False},
    'expectation_value': {'baseline': np.complex128(1+0j),
     'actual': np.complex128(1+0j),
     'ok': np.True_}}},
  {'detail': 'adjoint(compiler_pass(func))',
   'differences': {'gate_count': {'baseline': 6, 'actual': 4, 'ok': False},
    'expectation_value': {'baseline': np.complex128(1+0j),
     'actual': np.complex128(1+0j),
     'ok': np.True_}}}],
 'stage_2_differences': [{'detail': 'control(compiler_pass(adjoint(func)))',
   'differences': {'gate_count': {'baseline': 8, 'actual': 6, 'ok': False},
    'expectation_value': {'baseline': np.complex128(0.9999999999999996+0j),
     'actual': np.complex128(0.9999999999999996+0j),
     'ok': np.True_}}},
  {'detail': 'control(adjoint(compiler_pass(func))))',
   'differences': {'gate_count': {'baseline': 8, 'actual': 6, 'ok': False},
    'expectation_value': {'baseline': np.complex128(0.99999

Qiskit Bug - Qiskit - Zero Angle

In [6]:
from qiskit.circuit.parameter import Parameter
from qiskit.transpiler.passes import RemoveIdentityEquivalent


d = Parameter('d')

def target_function():
    qc = QuantumCircuit(2, 2)
    qc.ry(d * 0, 0)
    return qc

exp = Expectation(gate_count='decrease')

rq = RetroQ(target_function, compiler_pass=RemoveIdentityEquivalent, expectation=exp, framework='qiskit')

results = rq.run()
results

{'stage_1_differences': [{'detail': 'compiler_pass(adjoint(func))',
   'differences': {'gate_count': {'baseline': 2, 'actual': 2, 'ok': False},
    'expectation_value': {'baseline': np.complex128(1+0j),
     'actual': np.complex128(1+0j),
     'ok': np.True_}}},
  {'detail': 'adjoint(compiler_pass(func))',
   'differences': {'gate_count': {'baseline': 2, 'actual': 2, 'ok': False},
    'expectation_value': {'baseline': np.complex128(1+0j),
     'actual': np.complex128(1+0j),
     'ok': np.True_}}}],
 'stage_2_differences': [{'detail': 'control(compiler_pass(adjoint(func)))',
   'differences': {'gate_count': {'baseline': 4, 'actual': 4, 'ok': False},
    'expectation_value': {'baseline': np.complex128(0.9999999999999996+0j),
     'actual': np.complex128(0.9999999999999996+0j),
     'ok': np.True_}}},
  {'detail': 'control(adjoint(compiler_pass(func))))',
   'differences': {'gate_count': {'baseline': 4, 'actual': 4, 'ok': False},
    'expectation_value': {'baseline': np.complex128(0.99999

In [ ]:
import pennylane as qp

# 1. Define the target program (F)
def target_function():
    qp.Hadamard(wires=0)
    qp.Hadamard(wires=0)

# 2. Define user expectations for the compiler pass
# We expect the gate count to decrease
exp = Expectation(gate_count='decrease')

# 3. Initialize RetroQ with the pass to test
rq = RetroQ(target_function, compiler_pass=qp.transforms.cancel_inverses, expectation=exp, framework='pennylane')
rq.run()

In [8]:
results

{'stage_1_differences': [{'detail': 'compiler_pass(adjoint(func))',
   'differences': {'gate_count': {'baseline': 4, 'actual': 4, 'ok': False},
    'expectation_value': {'baseline': [np.float64(0.9999999999999993)],
     'actual': [np.float64(0.9999999999999993)],
     'ok': array([ True])}}}],
 'stage_2_differences': [{'detail': 'control(compiler_pass(adjoint(func))',
   'differences': {'gate_count': {'baseline': 6, 'actual': 6, 'ok': False},
    'expectation_value': {'baseline': np.float64(0.9999999999999996),
     'actual': np.float64(0.9999999999999993),
     'ok': np.True_}}}]}